In [1]:
import shutil
import zipfile
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# CBP Data

In [2]:
# Set paths
BASE = Path("data/raw/cbp")
RAW_DIR = BASE
OUT_DIR = BASE / "cbp_raw_1986_2023_parquet"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR_COUNTY = BASE / "cbp_county_thin_parquet"
OUT_DIR_DIST   = Path("data/intermediate/cbp_district_1986_2023_parquet")

OUT_DIR_COUNTY.mkdir(parents=True, exist_ok=True)
OUT_DIR_DIST.mkdir(parents=True, exist_ok=True)

shutil.rmtree(OUT_DIR_COUNTY, ignore_errors=True)
OUT_DIR_COUNTY.mkdir(parents=True, exist_ok=True)

shutil.rmtree(OUT_DIR_DIST, ignore_errors=True)
OUT_DIR_DIST.mkdir(parents=True, exist_ok=True)

## Helper Functions

In [3]:
# Extract year information from file name
def infer_year(zip_path: Path) -> int:
    yy = int(zip_path.stem.lower().replace("cbp", "").replace("co", ""))
    return 1900 + yy if yy >= 86 else 2000 + yy

In [4]:
# Map congressional district number and years
cd_year_map = [
    (102, 1986, 1991),
    (103, 1992, 1998),
    (106, 1999, 2002),
    (108, 2003, 2004),
    (109, 2005, 2008),
    (111, 2009, 2012),
    (113, 2013, 2014),
    (114, 2015, 2016),
    (115, 2017, 2018),
    (116, 2019, 2020),
    (117, 2021, 2022),
    (118, 2023, 2024),
    (119, 2025, 2026),
]

def congress_for_year(year: int) -> int:
    for cong, start, end in cd_year_map:
        if start <= year <= end:
            return cong

In [5]:
# Import and clean county-cd crosswalk data
def load_crosswalk_for_year(year: int) -> pd.DataFrame:
    cong = congress_for_year(year)
    path = Path("data/intermediate/county_to_cd_crosswalk.csv")

    cw = pd.read_csv(path, dtype="string")
    cw.columns = [c.strip().lower() for c in cw.columns]

    # filter to congress
    cw = cw[cw["cd" if "cd" in cw.columns else "congress"].astype(int) == cong]

    # normalize county keys
    cw["state_county_code"] = cw["state_county_code"].astype("string").str.zfill(5)
    cw["fipstate"] = cw["state_county_code"].str[:2]
    cw["fipscty"]  = cw["state_county_code"].str[2:]

    # normalize district + afact
    cw["district_code"] = cw["district_code"].astype("string").str.zfill(2)
    cw["afact"] = pd.to_numeric(cw["afact"], errors="coerce")

    return cw[["fipstate", "fipscty", "district_code", "afact"]]

In [6]:
# Map industries to broad sectors (primary/secondary/tertiary)
def industry_to_sector(code: str) -> str:
    if code is None or code == "":
        return "unknown"

    s = str(code)

    # distinguish between sic and naics
    digits_only = "".join(ch for ch in s if ch.isdigit())
    is_naics = len(digits_only) >= 5
    is_sic = len(digits_only) <= 4

    if len(digits_only) < 2:
        return "tertiary"

    d2 = int(digits_only[:2])

    # NAICS: primary = agriculture/mining; secondary = construction/manufacturing; tertiary = all service sectors
    if is_naics:
        if d2 in {11, 21}:
            return "primary"
        if d2 == 23 or 31 <= d2 <= 33:
            return "secondary"
        return "tertiary"

    # SIC: primary = agriculture/mining; secondary = construction/manufacturing; tertiary = all service sectors
    if is_sic:
        if 1 <= d2 <= 14:
            return "primary"
        if 15 <= d2 <= 17 or 20 <= d2 <= 39:
            return "secondary"
        return "tertiary"

    return "tertiary"

## Main Loop

In [7]:
# Save only helpful columns
KEEP_COLS = {"year", "fipstate", "fipscty", "industry", "emp", "est", "qp1", "ap"}

# Known unmatchable counties to drop
BAD = {
    ("12","025"),  # 2000
    ("08","014"),  # 2002
    ("02","280"), ("02","232"), ("02","201"),  # 2009
    ("02","261"),  # 2021
    ("09","001"), ("09","003"), ("09","005"), ("09","007"), ("09","009"),
    ("09","011"), ("09","013"), ("09","015"),  # 2021
    ("02","010"), ("02","140"),  # 1986
    ("02","068"),  # 1992
    ("02","282"),  # 1996
}
bad_mi = pd.MultiIndex.from_tuples(BAD, names=["fipstate", "fipscty"])

# Cache crosswalks by congress (so you don’t reread CSV each year)
cw_cache = {}

In [8]:
# Main loop to build, clean, and collapse dataset
for zip_path in sorted(RAW_DIR.glob("cbp*co.zip")):
    year = infer_year(zip_path)
    print(f"Processing {zip_path.name} → {year}")

    with zipfile.ZipFile(zip_path) as z:
        with z.open([m for m in z.namelist() if m.lower().endswith(".txt")][0]) as f:
            df = pd.read_csv(f, dtype="string[python]", low_memory=False)

    df.columns = [c.strip().lower() for c in df.columns]
    df["year"] = year

    # unify industry
    if "naics" in df.columns:
        df["industry"] = df["naics"]
    elif "sic" in df.columns:
        df["industry"] = df["sic"]
    else:
        raise ValueError(f"No industry code found for year {year}")

    df = df.drop(columns=["naics", "sic"], errors="ignore")

    # keep minimal columns
    df = df.loc[:, df.columns.intersection(KEEP_COLS)]

    # pad keys (before crosswalk merge)
    df["fipstate"] = df["fipstate"].astype("string").str.zfill(2)
    df["fipscty"]  = df["fipscty"].astype("string").str.zfill(3)

    # Drop state / non-county totals (CBP artifacts)
    df = df[df["fipscty"].notna()]
    df = df[~df["fipscty"].isin(["000", "999"])]

    # Drop your known problematic counties (compact)
    mi = pd.MultiIndex.from_frame(df[["fipstate", "fipscty"]])
    df = df[~mi.isin(bad_mi)]

    # Sector mapping AFTER drops (less work)
    df["sector"] = df["industry"].apply(industry_to_sector)

    # Cast numeric
    for c in ["emp", "est", "qp1", "ap"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Merge crosswalk (cached by congress)
    cong = congress_for_year(year)
    cw = cw_cache.get(cong)
    if cw is None:
        cw = load_crosswalk_for_year(year)
        cw_cache[cong] = cw

    df = df.merge(cw, on=["fipstate", "fipscty"], how="left")

    # Keep your existing hard-fail if anything is missing
    if df["district_code"].isna().any():
        miss = df["district_code"].isna().mean()
        raise ValueError(f"Missing district assignments in year={year}: {miss:.2%}")

    # Allocate and aggregate
    for c in ["emp", "est", "qp1", "ap"]:
        if c in df.columns:
            df[c] = df[c] * df["afact"]

    df["state_district_code"] = df["fipstate"] + df["district_code"]  # already zfilled in cw

    group_cols = ["year", "state_district_code", "sector"]
    agg_map = {c: "sum" for c in ["emp", "est", "qp1", "ap"] if c in df.columns}

    out = df.groupby(group_cols, as_index=False).agg(agg_map)

    pq.write_to_dataset(
        pa.Table.from_pandas(out, preserve_index=False),
        root_path=str(OUT_DIR_DIST),
        partition_cols=["year"]
    )

print("\n✅ Finished district-level parquet dataset")
print("District:", OUT_DIR_DIST)

Processing cbp00co.zip → 2000
Processing cbp01co.zip → 2001
Processing cbp02co.zip → 2002
Processing cbp03co.zip → 2003
Processing cbp04co.zip → 2004
Processing cbp05co.zip → 2005
Processing cbp06co.zip → 2006
Processing cbp07co.zip → 2007
Processing cbp08co.zip → 2008
Processing cbp09co.zip → 2009
Processing cbp10co.zip → 2010
Processing cbp11co.zip → 2011
Processing cbp12co.zip → 2012
Processing cbp13co.zip → 2013
Processing cbp14co.zip → 2014
Processing cbp15co.zip → 2015
Processing cbp16co.zip → 2016
Processing cbp17co.zip → 2017
Processing cbp18co.zip → 2018
Processing cbp19co.zip → 2019
Processing cbp20co.zip → 2020
Processing cbp21co.zip → 2021
Processing cbp22co.zip → 2022
Processing cbp23co.zip → 2023
Processing cbp86co.zip → 1986
Processing cbp87co.zip → 1987
Processing cbp88co.zip → 1988
Processing cbp89co.zip → 1989
Processing cbp90co.zip → 1990
Processing cbp91co.zip → 1991
Processing cbp92co.zip → 1992
Processing cbp93co.zip → 1993
Processing cbp94co.zip → 1994
Processing

## Data Export

In [9]:
# Read all parquet data files
df_all = pq.read_table(
    "data/intermediate/cbp_district_1986_2023_parquet"
).to_pandas()

In [10]:
# Add state and district code for merging
df_all['state_fips'] = df_all['state_district_code'].str[:2]
df_all['district_code'] = df_all['state_district_code'].str[2:]

# Map state fips to code
FIPS_TO_STATE = {
    '01': 'AL', '02': 'AK', '04': 'AZ', '05': 'AR', '06': 'CA',
    '08': 'CO', '09': 'CT', '10': 'DE', '11': 'DC', '12': 'FL',
    '13': 'GA', '15': 'HI', '16': 'ID', '17': 'IL', '18': 'IN',
    '19': 'IA', '20': 'KS', '21': 'KY', '22': 'LA', '23': 'ME',
    '24': 'MD', '25': 'MA', '26': 'MI', '27': 'MN', '28': 'MS',
    '29': 'MO', '30': 'MT', '31': 'NE', '32': 'NV', '33': 'NH',
    '34': 'NJ', '35': 'NM', '36': 'NY', '37': 'NC', '38': 'ND',
    '39': 'OH', '40': 'OK', '41': 'OR', '42': 'PA', '44': 'RI',
    '45': 'SC', '46': 'SD', '47': 'TN', '48': 'TX', '49': 'UT',
    '50': 'VT', '51': 'VA', '53': 'WA', '54': 'WV', '55': 'WI',
    '56': 'WY'
}
df_all['state_code'] = df_all['state_fips'].map(FIPS_TO_STATE)
df_all.drop(columns='state_fips', inplace=True)

In [11]:
# Save cleaned, district-level cbp data
df_all.to_csv("data/intermediate/cbp_district_level.csv", index=False)